# Products Data Cleaning

Problems to Investigate
- Misspelled column names
- Missing values in:
    - product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm
- Any duplicate product ids
- Leading/trailing whitespace in column names
- Leading/trailing whitespace in string values
- Inconsistent capitalization or formatting in product category names

In [2]:
import pandas as pd

products_raw = pd.read_csv('../data/raw/olist_products_dataset.csv')
products = products_raw.copy()

print("Original Shape:", products_raw.shape)
print("Working Shape:", products.shape)

Original Shape: (32951, 9)
Working Shape: (32951, 9)


In [6]:
# Fix misspelled column names

products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

print(products.dtypes)

product_id                        str
product_category_name             str
product_name_length           float64
product_description_length    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [15]:
# Investigate missing product category names, product name length, product description length,
# and product photo qty

missing_pattern = (
    products.isna()
    .value_counts()
    .reset_index(name="count")
)

missing_pattern

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,count
0,False,False,False,False,False,False,False,False,False,32340
1,False,True,True,True,True,False,False,False,False,609
2,False,False,False,False,False,True,True,True,True,1
3,False,True,True,True,True,True,True,True,True,1


In [16]:
# Investigate products missing category name 

missing_category = products[
    products["product_category_name"].isna()
]

print("Products missing category:", len(missing_category))
display(missing_category.head(10))

Products missing category: 610


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [19]:
# Check whether products missing categories have been sold

order_items = pd.read_csv('../data/processed/olist_order_items_cleaned.csv')

missing_category_orders = missing_category.merge(
    order_items[["product_id", "order_id"]],
    on="product_id",
    how="left",
    indicator=True
)

print(missing_category_orders["_merge"].value_counts())

print(
    "\nUnique products with missing categories that were ordered:",
    missing_category_orders["product_id"].nunique()
)

_merge
both          1603
left_only        0
right_only       0
Name: count, dtype: int64

Unique products with missing categories that were ordered: 610


In [20]:
# Investigate products missing dimensions

missing_dimensions = products[
    products["product_length_cm"].isna()
]

print("Products missing dimensions:", len(missing_dimensions))
display(missing_dimensions.head())

Products missing dimensions: 2


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Check whether products missing dimensions have been sold

missing_dimension_orders = missing_dimensions.merge(
    order_items[["product_id", "order_id"]],
    on="product_id",
    how="left",
    indicator=True
)

print(missing_dimension_orders["_merge"].value_counts())

print(
    "\nUnique products with missing dimensions that were ordered:",
    missing_dimension_orders["product_id"].nunique()
)

_merge
both          18
left_only      0
right_only     0
Name: count, dtype: int64

Unique products with missing dimensions that were ordered: 2


In [27]:
# Flag products missing descriptive attributes
description_columns = [
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty"
]

products["missing_descriptions"] = (
    products[description_columns].isna().any(axis=1)
)

# Flag products missing physical dimensions
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products["missing_dimensions"] = (
    products[dimension_columns].isna().any(axis=1)
)

print("Products missing descriptions:", products["missing_descriptions"].sum())
print("Products missing dimensions:", products["missing_dimensions"].sum())

Products missing descriptions: 610
Products missing dimensions: 2


In [32]:
# Check for duplicate product ids 

duplicate_ids = products["product_id"].duplicated().sum()

print("Duplicate product IDs:", duplicate_ids)


Duplicate product IDs: 0


In [33]:
# Checking for whitespace

string_columns = products.select_dtypes(include=["object", "str"]).columns

for column in string_columns:
    count = products[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {count} values with whitespace")

products.columns = (
    products.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(products.columns.tolist())

product_id: 0 values with whitespace
product_category_name: 0 values with whitespace
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'missing_descriptions', 'missing_dimensions']


In [35]:
# Checking for any inconsistent capitalizations in category names

print("Number of unique categories:", products["product_category_name"].nunique())

print(
    products["product_category_name"]
    .dropna()
    .sort_values()
    .unique()
)

category_case_check = (
    products.dropna(subset=["product_category_name"])
    .groupby(products["product_category_name"].str.lower())[
        "product_category_name"
    ]
    .nunique()
)

print("Potential capitalization inconsistencies:")
display(category_case_check[category_case_check > 1])

Number of unique categories: 73
<StringArray>
[                     'agro_industria_e_comercio',
                                      'alimentos',
                              'alimentos_bebidas',
                                          'artes',
                             'artes_e_artesanato',
                              'artigos_de_festas',
                               'artigos_de_natal',
                                          'audio',
                                     'automotivo',
                                          'bebes',
                                        'bebidas',
                                   'beleza_saude',
                                     'brinquedos',
                                'cama_mesa_banho',
                                  'casa_conforto',
                                'casa_conforto_2',
                                'casa_construcao',
                              'cds_dvds_musicais',
                                    

Series([], Name: product_category_name, dtype: int64)

In [37]:
products.to_csv(
    "../data/processed/olist_products_cleaned.csv",
    index=False
)